In [1]:
import pandas as pd
df = pd.read_csv("data/processed_dataset.csv")

In [2]:
print(df["Application_Text"].sample(5).tolist())

['Krishna Mehta, based in Amritsar, is requesting a auto loan amounting to INR 420,976. The applicant mentioned wanting to avoid delays seen in a previous application. The applicant has asked whether a co-signer will be required for approval. The applicant referenced salaried employment status in the form.', 'Samina Ansari, based in New Delhi, is requesting a personal loan amounting to INR 127,822. Funds are needed to consolidate existing high-interest debt. They have asked for expedited processing given the time-sensitive nature of the request. This is stated to be their first loan application with the bank.', 'Kavya Iyer, based in Lucknow, is requesting a medical loan amounting to INR 178,489. The applicant cited a family event as the primary driver for this request. The applicant asked for an estimated disbursal timeline. No prior loan history was mentioned in the application.', 'Ishita Singh approached HDFC Bank seeking a home loan of INR 2,657,792. The applicant mentioned wanting 

In [3]:
print(df[[
    "Loan_Status",
    "CIBIL_Score",
    "Credit_History",
    "Default_History_Count",
    "Debt_to_Income_Ratio",
    "Application_Text"
]].head(10))

  Loan_Status  CIBIL_Score  Credit_History  Default_History_Count  \
0    Approved          699               1                      0   
1    Rejected          707               1                      0   
2    Approved          641               1                      0   
3    Approved          677               1                      0   
4    Approved          594               0                      0   
5    Approved          682               1                      0   
6    Approved          693               1                      0   
7    Rejected          723               1                      0   
8    Rejected          593               0                      0   
9    Approved          538               0                      0   

   Debt_to_Income_Ratio                                   Application_Text  
0                 0.098  Rohan Verma approached HDFC Bank seeking a hom...  
1                 4.939  This is a business loan request from Rohan Ver...  
2        

In [4]:
print(df["Loan_Status"].value_counts())

Loan_Status
Approved    653
Rejected    347
Name: count, dtype: int64


In [5]:
print(df["Application_Text"].nunique())

1000


In [6]:
def create_bert_text(row):
    return f"""
Applicant Information:
Age: {row['Age']}
Gender: {row['Gender']}
Married: {row['Married']}
Dependents: {row['Dependents']}
Education: {row['Education']}
Occupation: {row['Occupation']}
Employment Status: {row['Employment_Status']}
Employment Length: {row['Employment_Length_Years']} years

Financial Information:
Applicant Income: {row['Applicant_Income']}
Co-applicant Income: {row['Coapplicant_Income']}
Annual Household Income: {row['Annual_Household_Income']}
Monthly Expense: {row['Monthly_Expense']}
Debt to Income Ratio: {row['Debt_to_Income_Ratio']}

Credit Information:
CIBIL Score: {row['CIBIL_Score']}
Credit History: {"Positive repayment history" if row['Credit_History'] == 1 else "Poor repayment history"}
Previous Loans: {row['Number_of_Previous_Loans']}
Default History: {row['Default_History_Count']}
Existing EMIs: {row['Existing_EMIs']}

Loan Information:
Purpose of Loan: {row['Purpose_of_Loan']} Loan
Loan Amount: {row['Loan_Amount']}
Loan Term: {row['Loan_Term_Months']} months

Assets:
Asset Value: {row['Asset_Value']}
Guarantor Available: {"Yes" if row['Guarantor'] == 1 else "No"}

Location:
Property Area: {row['Property_Area']}
State: {row['State']}
City: {row['City']}
Region Branch: {row['Region_Branch']}

Customer Relationship:
Institutional Relationships: {row['Institutional_Relationships']}
""".strip()

In [7]:
df["Application_Text_BERT"] = df.apply(create_bert_text, axis=1)

In [8]:
df.to_csv("data/processed_dataset.csv", index=False)

print("Application_Text_BERT column created successfully!")

print("\nExample:\n")
print(df["Application_Text_BERT"].iloc[0])

Application_Text_BERT column created successfully!

Example:

Applicant Information:
Age: 36
Gender: Male
Married: No
Dependents: 2
Education: Graduate
Occupation: Farmer
Employment Status: Salaried
Employment Length: 1 years

Financial Information:
Applicant Income: 56976
Co-applicant Income: 0
Annual Household Income: 683712
Monthly Expense: 30391
Debt to Income Ratio: 0.098

Credit Information:
CIBIL Score: 699
Credit History: Positive repayment history
Previous Loans: 0
Default History: 0
Existing EMIs: 5610

Loan Information:
Purpose of Loan: Home Loan
Loan Amount: 8031545
Loan Term: 360 months

Assets:
Asset Value: 744861
Guarantor Available: No

Location:
Property Area: Urban
State: Delhi
City: Dwarka
Region Branch: KOL-004

Customer Relationship:
Institutional Relationships: RBI:Regulatory, NPCI:Payments integration, CIBIL:Credit bureau


In [9]:
print(df["Loan_Status"].value_counts())

print(df.groupby("Loan_Status")["CIBIL_Score"].describe())

Loan_Status
Approved    653
Rejected    347
Name: count, dtype: int64
             count        mean        std    min    25%    50%    75%    max
Loan_Status                                                                 
Approved     653.0  666.833078  66.360870  484.0  623.0  665.0  714.0  878.0
Rejected     347.0  627.005764  80.023587  384.0  576.0  627.0  680.0  853.0


In [10]:
print(pd.crosstab(df["Credit_History"], df["Loan_Status"]))

Loan_Status     Approved  Rejected
Credit_History                    
0                    117       160
1                    536       187


In [11]:
print(pd.crosstab(df["Default_History_Count"], df["Loan_Status"]))

Loan_Status            Approved  Rejected
Default_History_Count                    
0                           621       285
1                            21        29
2                            10        25
3                             1         8


In [12]:
print(df["Loan_Status"].value_counts(normalize=True))

Loan_Status
Approved    0.653
Rejected    0.347
Name: proportion, dtype: float64


In [13]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

lengths = [
    len(tokenizer.tokenize(x))
    for x in df["Application_Text_BERT"]
]

print("Max:", max(lengths))
print("Mean:", sum(lengths)/len(lengths))
print("95th percentile:", np.percentile(lengths,95))

C:\Users\Admin\DataspellProjects\Intelligent-Loan-Intelligence-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Max: 189
Mean: 178.778


NameError: name 'np' is not defined